# Comparaison des métriques des modèles GP

Ce notebook compare les métriques (ESS, Rhat, RMSE, MAE) pour tous les modèles dans `Output/gaussian_process/`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from sklearn.metrics import  mean_absolute_error, root_mean_squared_error
import re
import numpy as np

In [9]:
# TCD 

path_tcd = [
    Path("/Users/kopp/Documents/efficientGP/output/gaussian_process/TCD_gp_with_features_ridge_centered_20251216_114735"),
    Path("/Users/kopp/Documents/efficientGP/output/data/tcd_bayesian_ridge"),
    Path("/Users/kopp/Documents/efficientGP/output/data/tcd_mlp"),
    Path("/Users/kopp/Documents/efficientGP/output/gaussian_process/TCD_gp_no_prior_20251210_112502"),
    Path("/Users/kopp/Documents/efficientGP/output/data/tcd_xg_boost")
    ]

# Nigeria 
path_nga = [
    Path("/Users/kopp/Documents/efficientGP/output/gaussian_process/NGA_gp_with_features_normal_centered_20260217_155435"),
    Path("/Users/kopp/Documents/efficientGP/output/data/nga_bayesian_ridge"),
    Path("/Users/kopp/Documents/efficientGP/output/data/nga_mlp"),
    Path("/Users/kopp/Documents/efficientGP/output/data/nga_xg_boost"),
    Path("/Users/kopp/Documents/efficientGP/output/gaussian_process/NGA_gp_no_features_20251020_161159")
    ] 


# Functions

In [10]:
def load_model_metrics_stat(model_folder: Path):
    model = str(model_folder).split('/')[-1].split('_')
    gp_index = model.index('gp')
    model_name = ' '.join(model[gp_index : (gp_index+3)]).title()
    
    ppc_summary_path = model_folder / "prior_posterior_diagnostics" / "ppc_summary_all_folds.csv"
    
    if ppc_summary_path.exists():
        metrics = (
            pd.read_csv(ppc_summary_path)
            .query("cv_fold != 0")
            .groupby(['cv_fold', 'adm1_name'])[['y_true', 'y_pred_mean', 'y_pred_ci_2.5', 'y_pred_ci_97.5', 'len_ic_95']]
            .apply(
                lambda x: pd.Series({
                    'RMSE': root_mean_squared_error(x['y_true'].values, x['y_pred_mean'].values),
                    'MAE': mean_absolute_error(x['y_true'].values, x['y_pred_mean'].values),
                    'Coverage 95': ((x['y_true'].values >= x['y_pred_ci_2.5'].values) & (x['y_true'].values <= x['y_pred_ci_97.5'].values)).mean(),
                    'Length IC 95': x['len_ic_95'].mean()
                })
            )
            .reset_index()
            .groupby('adm1_name')[['RMSE', 'MAE', 'Coverage 95', 'Length IC 95']]
            .mean()
            .reset_index()
            .assign(Model=model_name)
        )
        
        #print(metrics)
        
        return metrics

In [11]:
def load_model_metrics_mlp(model_folder: Path):
    all_metrics = []
    
    for i in range(1, 6):     
        df = pd.read_csv(os.path.join(model_folder, f'predictions_cv{i}.csv'))
        df = df[['y_hat', 'y_true', 'ci_low', 'ci_high', 'adm1_name']]
        
        metrics = (
            df
            .groupby('adm1_name')[['y_hat', 'y_true', 'ci_low', 'ci_high', 'adm1_name']]
            .apply(
                lambda x: pd.Series({
                    'RMSE': root_mean_squared_error(x['y_true'], x['y_hat']),
                    'MAE': mean_absolute_error(x['y_true'], x['y_hat']),
                    'Coverage 95': ((x['y_true'] >= x['ci_low']) & (x['y_true'] <= x['ci_high'])).mean(),
                    'Length IC 95': (x['ci_high'] - x['ci_low']).mean()
                    })
            )
        )
        all_metrics.append(metrics)
    
    return (pd
            .concat(all_metrics)
            .mean()
            .pipe(lambda s: pd.DataFrame([s]))
            .assign(Model='MLP')
            )

In [12]:
def load_model_metrics_br(model_folder: Path):
    all_metrics = []
    country = str(model_folder).split('/')[-1].split('_')[0]

    for i in range(1, 6):   
        df = pd.read_csv(os.path.join(model_folder, f'{country}_predictions_fold{i}.csv'))
        
        metrics = (
            df.groupby('adm1_name')[['y_hat', 'y_true', 'ci_low', 'ci_high', 'adm1_name']]
            .apply(
            lambda x: pd.Series({
                'RMSE': root_mean_squared_error(x['y_true'], x['y_hat']),
                'MAE': mean_absolute_error(x['y_true'], x['y_hat']),
                'Coverage 95': ((x['y_true'] >= x['ci_low']) & (x['y_true'] <= x['ci_high'])).mean(),
                'Length IC 95': (x['ci_high'] - x['ci_low']).mean()
            })
        ))
        all_metrics.append(metrics)
    
    return (pd
            .concat(all_metrics)
            .mean()
            .pipe(lambda s: pd.DataFrame([s]))
            .assign(Model='Bayesian Ridge')
            )

In [13]:
def load_model_metrics_xgboost(model_folder: Path):
    all_metrics = []
    
    for i in range( 1, 6):     
        df = (pd
            .read_csv(os.path.join(model_folder, "predictions_cv{}.csv".format(i)))
        )
        
        metrics = (
            df.groupby('adm1_name')[['y_hat', 'y_true', 'ci_low', 'ci_high']]
            .apply(
                lambda x: pd.Series({
                    'RMSE': root_mean_squared_error(x['y_true'], x['y_hat']),
                    'MAE': mean_absolute_error(x['y_true'], x['y_hat']),
                    'Coverage 95': ((x['y_true'] >= x['ci_low']) & (x['y_true'] <= x['ci_high'])).mean(),
                    'Length IC 95': (x['ci_high'] - x['ci_low']).mean()
                })
        ))
        all_metrics.append(metrics)
    
    return  pd.concat(all_metrics).assign(Model='XG-Boost').reset_index()

# Model comparison

## Nigeria 

In [14]:
# Nigeria 
df_gp_cov = load_model_metrics_stat(path_nga[0])
df_mlp = load_model_metrics_mlp(path_nga[2])
df_br = load_model_metrics_br(path_nga[1])
df_xg_boost = load_model_metrics_xgboost(path_nga[3])
df_gp_no_cov = load_model_metrics_stat(path_nga[4])


In [17]:
df_all = pd.concat([df_gp_cov, df_mlp, df_br, df_xg_boost, df_gp_no_cov], ignore_index=True)
#df_all = df_all.round(3)
#df_all['Model'] = df_all['Model'].str.replace('Gp', 'GP')
df_all = df_all.groupby('Model')[["RMSE","MAE","Coverage 95","Length IC 95"]].mean().reset_index()

In [18]:
df_all

,Model,RMSE,MAE,Coverage 95,Length IC 95
0,Bayesian Ridge,0.052441,0.044651,0.703175,0.111518
1,Gp No Features,0.074033,0.069047,1.000000,0.475831
2,Gp With Features,0.047309,0.041609,0.994444,0.338572
3,MLP,0.059277,0.050997,0.757143,0.169454
4,XG-Boost,0.056757,0.049336,0.279365,0.038538


In [ ]:
df_all.to_latex(
    "/Users/kopp/Documents/efficientGP/output/tables/nga_prediction_metrics.tex",
    index=False,
    caption='Metrics for Nigeria models',
    label='nga-tab-models',
    float_format='%.3f'
)

## Chad

In [20]:
# Chad
df_gp_cov = load_model_metrics_stat(path_tcd[0])
df_gp_cov = df_gp_cov[['Model', 'RMSE', 'MAE', 'Coverage 95', 'Length IC 95']]

df_mlp = load_model_metrics_mlp(path_tcd[2])
df_br = load_model_metrics_br(path_tcd[1])
df_gp_no_cov = load_model_metrics_stat(path_tcd[3])
df_xg_boost = load_model_metrics_xgboost(path_tcd[4])


In [21]:
df_all = pd.concat([df_gp_cov, df_mlp, df_br, df_gp_no_cov, df_xg_boost], ignore_index=True)
df_all = (
    df_all.groupby('Model')[["RMSE","MAE","Coverage 95","Length IC 95"]]
    .mean()
    .reset_index()
    .replace('Gp No Prior', 'GP no features')
    )

In [22]:
df_all

,Model,RMSE,MAE,Coverage 95,Length IC 95
0,Bayesian Ridge,0.064877,0.054932,0.600529,0.116186
1,GP no features,0.052918,0.046183,0.965608,0.286568
2,Gp With Features,0.051683,0.044767,0.959436,0.243424
3,MLP,0.053317,0.044930,0.731041,0.133049
4,XG-Boost,0.059388,0.052558,0.190476,0.029631


In [ ]:
df_all.to_latex(
    "/Users/kopp/Documents/efficientGP/output/tables/tcd_prediction_metrics.tex",
    index=False,
    caption='Metrics for Chad models',
    label='tcd-tab-models',
    float_format='%.3f'
)